In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Automatically detect target column (last column)
target_col = df.columns[-1]
print("Detected Target Column:", target_col)

# Encode categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
encoders = {}

for col in cat_cols:
    enc = LabelEncoder()
    df[col] = enc.fit_transform(df[col].astype(str))
    encoders[col] = enc

# Train-test split
X = df.drop(target_col, axis=1)
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Detected Target Column: Churn


In [ ]:
# Logistic Regression baseline from sklearn.linear_model import LogisticRegression from sklearn.metrics import accuracy_score, classification_report log_model = LogisticRegression(max_iter=500) log_model.fit(X_train, y_train) y_pred_log = log_model.predict(X_test) print("📌 Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log)) print("\nClassification Report:\n", classification_report(y_test, y_pred_log)) for this
# Cell 2: Logistic Regression baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("📌 Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))



📌 Logistic Regression Accuracy: 0.8140525195173882

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.67      0.58      0.62       373

    accuracy                           0.81      1409
   macro avg       0.76      0.74      0.75      1409
weighted avg       0.81      0.81      0.81      1409



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# Cell 3: Decision Tree baseline
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))


📌 Decision Tree Accuracy: 0.794180269694819

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.96      0.87      1036
           1       0.74      0.34      0.47       373

    accuracy                           0.79      1409
   macro avg       0.77      0.65      0.67      1409
weighted avg       0.79      0.79      0.77      1409



In [ ]:
# Cell 4: Show combined comparison
print("Baseline Model Results:")
print("---------------------------------------------------")
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nObservation: Both simple models show low performance,")
print("which motivated the use of advanced models like XGBoost, LightGBM, CatBoost, and Optuna Tuning.")


Baseline Model Results:
---------------------------------------------------
Logistic Regression Accuracy: 0.8140525195173882
Decision Tree Accuracy: 0.794180269694819

Observation: Both simple models show low performance,
which motivated the use of advanced models like XGBoost, LightGBM, CatBoost, and Optuna Tuning.


In [ ]:
#ChurnShield
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

!pip install -q pandas numpy scikit-learn xgboost catboost optuna imbalanced-learn joblib matplotlib seaborn

import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.ensemble import VotingClassifier
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler
import matplotlib.pyplot as plt, seaborn as sns
import joblib
from google.colab import files


DATA_PATH = "/content/WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = "/content/models"; OUTPUT_DIR = "/content/outputs"
os.makedirs(MODEL_DIR, exist_ok=True); os.makedirs(OUTPUT_DIR, exist_ok=True)


df = pd.read_csv(DATA_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True, errors='ignore')

df['TenureGroup'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 72],labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr'])
df['AvgMonthlyCharges'] = df['TotalCharges'] / (df['tenure'] + 1)
df['LongTenure'] = (df['tenure'] >= 48).astype(int)
svc_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df['NumAddOns'] = df[svc_cols].apply(lambda r: sum(v=='Yes' for v in r), axis=1)

y = df['Churn'].map({'Yes':1, 'No':0})
X = df.drop(columns=['Churn'])
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(include=np.number).columns

ct = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])
X_enc = ct.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_enc, y, test_size=0.2, stratify=y, random_state=1)
X_train_s, y_train_s = SMOTE(random_state=1).fit_resample(X_train, y_train)

def tune_xgb(X, y):
    def obj(trial):
        params = dict(
            n_estimators=trial.suggest_int('n_estimators',200,500),
            max_depth=trial.suggest_int('max_depth',3,8),
            learning_rate=trial.suggest_float('learning_rate',0.05,0.2),
            subsample=trial.suggest_float('subsample',0.7,1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree',0.7,1.0),
            random_state=1, eval_metric='logloss', use_label_encoder=False
        )
        model = XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
        return cross_val_score(model, X, y, cv=cv, scoring='f1').mean()
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=1))
    study.optimize(obj, n_trials=10, show_progress_bar=False)
    print("Best XGB:", study.best_params)
    return XGBClassifier(**study.best_params, random_state=1, eval_metric='logloss', use_label_encoder=False)

def tune_cat(X, y):
    def obj(trial):
        params = dict(
            iterations=trial.suggest_int('iterations',200,500),
            depth=trial.suggest_int('depth',4,8),
            learning_rate=trial.suggest_float('learning_rate',0.05,0.2),
            random_state=1, verbose=0
        )
        model = CatBoostClassifier(**params)
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
        return cross_val_score(model, X, y, cv=cv, scoring='f1').mean()
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=1))
    study.optimize(obj, n_trials=10, show_progress_bar=False)
    print("Best CatBoost:", study.best_params)
    return CatBoostClassifier(**study.best_params, random_state=1, verbose=0)

print("Tuning XGBoost...")
xgb_model = tune_xgb(X_train_s, y_train_s)
print("Tuning CatBoost...")
cat_model = tune_cat(X_train_s, y_train_s)

xgb_model.fit(X_train_s, y_train_s)
cat_model.fit(X_train_s, y_train_s)
voting = VotingClassifier([('xgb', xgb_model), ('cat', cat_model)], voting='soft')
voting.fit(X_train_s, y_train_s)

proba = voting.predict_proba(X_test)[:,1]
thr = np.arange(0.3,0.8,0.01)
f1s = [f1_score(y_test, proba>=t) for t in thr]
best_thr = thr[np.argmax(f1s)]
pred = (proba>=best_thr).astype(int)
print(f"\nBest threshold: {best_thr:.2f}")
print(classification_report(y_test, pred))
print("Accuracy:", accuracy_score(y_test, pred), "F1:", f1_score(y_test, pred))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", bbox_inches='tight')
plt.close()

package = {'model': voting, 'preprocessor': ct, 'threshold': float(best_thr)}
joblib.dump(package, f"{MODEL_DIR}/churn_model.pkl")
print("\n Model saved at:", f"{MODEL_DIR}/churn_model.pkl")

files.download(f"{MODEL_DIR}/churn_model.pkl")
files.download(f"{OUTPUT_DIR}/confusion_matrix.png")




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 24.9 MB/s eta 0:00:00


[I 2025-11-26 06:33:41,097] A new study created in memory with name: no-name-ca7f255b-1df2-4833-972d-9757e38781a1


Tuning XGBoost...


[I 2025-11-26 06:33:49,608] Trial 0 finished with value: 0.8533703265589229 and parameters: {'n_estimators': 325, 'max_depth': 7, 'learning_rate': 0.050017156222601736, 'subsample': 0.7906997717895519, 'colsample_bytree': 0.7440267672451338}. Best is trial 0 with value: 0.8533703265589229.
[I 2025-11-26 06:33:51,449] Trial 1 finished with value: 0.8515129476610633 and parameters: {'n_estimators': 227, 'max_depth': 4, 'learning_rate': 0.10183410905645718, 'subsample': 0.819030242269201, 'colsample_bytree': 0.8616450202010071}. Best is trial 0 with value: 0.8533703265589229.
[I 2025-11-26 06:33:58,667] Trial 2 finished with value: 0.849372919038359 and parameters: {'n_estimators': 326, 'max_depth': 7, 'learning_rate': 0.08066783745972762, 'subsample': 0.9634352309172836, 'colsample_bytree': 0.7082162779593778}. Best is trial 0 with value: 0.8533703265589229.
[I 2025-11-26 06:34:03,061] Trial 3 finished with value: 0.8457157840129444 and parameters: {'n_estimators': 401, 'max_depth': 5, '

Best XGB: {'n_estimators': 469, 'max_depth': 3, 'learning_rate': 0.05585821748493236, 'subsample': 0.7509491258693707, 'colsample_bytree': 0.9634427510288239}
Tuning CatBoost...


...

Best CatBoost: {'iterations': 261, 'depth': 8, 'learning_rate': 0.05410813897968893}

Best threshold: 0.39
              precision    recall  f1-score   support

           0       0.89      0.79      0.84      1033
           1       0.56      0.73      0.63       374

    accuracy                           0.77      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.80      0.77      0.78      1407

Accuracy: 0.7739872068230277 F1: 0.6310904872389791

 Model saved at: /content/models/churn_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>